# LLM Evaluation 3 of 4: Which Claim Does the Passage Support?

The award answer sounded plausible and shared many source words. Asking for one overall
score still left the editor with a practical question: **what, exactly, should I check?**

Our passage says Elena Marchetti was **longlisted** for a prize.
The practice answer says she **won**, then adds a book title the passage never mentions.
We need to point to those particular claims and the evidence available for them.

This notebook opens a deliberately simple rule-based attempt. It does not run a pretrained
factuality model. We will make its failures visible rather than turn its labels into
claims about detector accuracy.

### From a suspicious answer to a reviewable question

| What do we need to know? | What we try here | What remains limited |
| --- | --- | --- |
| Which part of the answer needs checking? | Split text into short pieces | A short piece can still contain several facts |
| Is a name or relationship unsupported? | Compare word groups, names and years | Word matching is not a proof of support |
| Which cases should go to review? | Try several score cutoffs against declared labels | This tiny practice set cannot validate a real catch rate |

A **claim** is something we could check, such as a prize outcome or publication date.
**Grounded** means supported by the supplied evidence; it does not mean independently
verified true in the world. The source itself could be wrong or incomplete.

In [ ]:
# Deterministic setup and the shared Riverside evidence fixture.
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from evaluation_shared import COURSE_CAVEAT, RIVERSIDE_CASES, normalized_terms, set_seed, tokenize

set_seed()
print(COURSE_CAVEAT)


## Part 1 - One sentence can hide several things to check

Separate the questions an editor would ask:

| Detail in the practice answer | What the passage establishes |
| --- | --- |
| Person, prize and year | Present in the source |
| `won` | Not established by `longlisted` |
| The added novel title | Not supplied |

The next function is our crude first attempt: split at full stops, semicolons and `and`.
**Predict:** will that actually separate the award outcome from the book title in this sentence?

In [ ]:
# Small claim splitter for the controlled fixture.
def split_claims(answer):
    pieces = re.split(r"[.;]|\band\b", answer)
    return [piece.strip() for piece in pieces if piece.strip()]

award = RIVERSIDE_CASES[1]
answer = award["fluent_wrong"]
claims = split_claims(answer)
display(pd.DataFrame({"claim": claims}))
print("Evidence:", award["context"])


In [ ]:
# Outcome rules expose support, contradiction, and missing evidence.
CONTRADICTIONS = {
    ("automatic", "requires"),
    ("allows", "prohibited"),
}

def claim_outcome(claim, evidence):
    claim_terms, evidence_terms = set(normalized_terms(claim)), set(normalized_terms(evidence))
    # A longlist does not establish either a win or a loss.
    if "won" in claim_terms and "longlist" in evidence_terms:
        return "unknown"
    for claim_term, evidence_term in CONTRADICTIONS:
        if claim_term in claim_terms and evidence_term in evidence_terms:
            return "contradicted"
    coverage = len(claim_terms & evidence_terms) / max(len(claim_terms), 1)
    return "supported" if coverage >= 0.6 else "unknown"

claim_rows = [
    {"claim": claim, "outcome": claim_outcome(claim, award["context"])}
    for claim in claims
]
claim_table = pd.DataFrame(claim_rows)
display(claim_table)


### Three evidence outcomes, not three degrees of certainty

| Intended outcome | Plain-English meaning |
| --- | --- |
| Supported | The evidence establishes the claim |
| Contradicted | The evidence establishes something incompatible |
| Unknown | The supplied evidence does not settle it |

Those definitions describe the job we want done. The function above uses a few chosen word
pairs and shared-word coverage as shortcuts. Its labels must not be mistaken for logical proof.

### We found a review target, not a complete claim extractor

The award sentence remains **one piece**. It still combines the prize outcome and novel title.
The split rule did not recover all the checkable facts; a real claim extractor or human
would need to separate them more carefully.

`unknown` is the right support question for the stronger `won` claim:
being longlisted proves neither winning nor not winning.
The title also needs evidence; its absence here does not prove that no such book exists.

The next labels come from small word rules and a coverage cutoff.
Even `supported` means only that the toy's rule passed; it is not a validated support verdict.
A rule-triggered `contradicted` label also needs checking against the actual wording.

## Part 2 - Point the editor to the missing detail

First ask which names, years or titles appeared in the answer but not the passage.
Such items are often called **entities**. Then ask a different question:
did the answer change the relationship between familiar entities?

`Elena`, the prize and `2019` can all match while `longlisted` becomes `won`.
A name check alone cannot catch that change.

Our next extractor only collects capitalized word groups and four-digit years.
It is a convenient practice rule, not a full name/date recognizer.

In [ ]:
# Extract capitalized spans and years as inspectable entity candidates.
def entities(text):
    names = re.findall(r"\b(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b", text)
    years = re.findall(r"\b(?:19|20)\d{2}\b", text)
    return set(names + years)

entity_gap = entities(answer) - entities(award["context"])
relation_pairs = [
    {"claim_relation": "won", "evidence_relation": "longlisted", "match": False},
]
print("entity_gap=", sorted(entity_gap))
display(pd.DataFrame(relation_pairs))


### The title flag is helpful; the extraction is still crude

For the award answer, the extractor returns `Bridges` and `The Silence`, not one clean
novel-title span. Elsewhere it even flags the sentence-opening word `In`.

The printed relation row separately compares `won` with `longlisted`.
That row is written for this example; no model discovered a general relation-matching ability.
Use both clues to focus the review, not to rubber-stamp the answer.

In [ ]:
# Retain the evidence span and signal provenance for every claim.
localized_rows = []
for case in RIVERSIDE_CASES:
    for answer_type in ["candidate", "fluent_wrong"]:
        for claim in split_claims(case[answer_type]):
            outcome = claim_outcome(claim, case["context"])
            localized_rows.append({
                "case": case["id"],
                "answer_type": answer_type,
                "claim": claim,
                "evidence_span": case["context"],
                "outcome": outcome,
                "entity_gap": sorted(entities(claim) - entities(case["context"])),
            })
localized = pd.DataFrame(localized_rows)
display(localized[["case", "answer_type", "claim", "outcome", "entity_gap"]])


Each row keeps its source passage with the text being checked, so a reviewer can revisit
the evidence. The simple splitter even makes `calculation` a standalone ReAct fragment.
That is a visible warning: rows in this table are not guaranteed to be complete atomic claims.

## Part 3 - How much review are we willing to do?

Review every answer and the workload is large. Review only the most suspicious answers
and some problems may slip through. We need to see both costs.

The code assigns chosen scores to its provisional outcomes, then adds a small amount
when the name check finds a gap. These scores are **not probabilities of hallucination**.
They are a simple way to put practice rows in order.

| Reading | Question |
| --- | --- |
| Precision | Of the rows sent for review, how many carry the declared problem label? |
| Recall | Of the declared problem rows, how many did we send? |
| Review rate | What fraction of all rows did we send? |

The label is inherited from `fluent_wrong` for every fragment of that answer.
Those are not independently reviewed claim labels. A clean-looking result here therefore
does not establish the performance of a real detector.

In [ ]:
# Convert observed claim outcomes into a review-risk score.
def risk_score(row):
    base = {"supported": 0.05, "unknown": 0.55, "contradicted": 0.90}[row["outcome"]]
    return min(1.0, base + 0.10 * bool(row["entity_gap"]))

localized["risk"] = localized.apply(risk_score, axis=1)
localized["label"] = localized["answer_type"].eq("fluent_wrong")

threshold_rows = []
for threshold in np.linspace(0.1, 0.9, 9):
    predicted = localized["risk"] >= threshold
    true_positive = int((predicted & localized["label"]).sum())
    false_positive = int((predicted & ~localized["label"]).sum())
    false_negative = int((~predicted & localized["label"]).sum())
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    threshold_rows.append({
        "threshold": threshold, "precision": precision, "recall": recall,
        "review_rate": predicted.mean(),
    })
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(3))


In [ ]:
# Static threshold evidence: recall and review load move together.
ax = threshold_table.plot(
    x="threshold", y=["precision", "recall", "review_rate"],
    marker="o", figsize=(9, 4)
)
ax.set_ylim(0, 1.05)
ax.set_title("Claim-risk threshold trade-off")
ax.set_ylabel("rate")
plt.tight_layout()
plt.show()

eligible = threshold_table.query("recall >= 0.80")
chosen = eligible.sort_values(["review_rate", "threshold"]).iloc[0]
print("Chosen teaching threshold:", chosen.to_dict())


### Read the selected row in the context of this practice set

At the chosen cutoff **0.2**, four of nine fragments are sent for review.
All four carry the declared wrong-answer label, so this fixture reports precision and recall
of **1.0**. That is not a measured 100% catch rate on future claims.

The `0.80` and `0.90` recall requirements select the same row: both require the available
1.0-recall level. The score grid jumps; no extra review is needed for that particular change.

## Your turn - can a stricter target leave the choice unchanged?

1. **Predict:** the available recall values jump between a few levels. Will moving the
   required recall from `0.80` to `0.90` necessarily require more review?
2. **Change:** alter only that condition in the `eligible` query.
3. **Measure:** rerun the selection and record threshold, recall, precision and review rate.
4. **Diagnose:** if the choice stays the same, inspect which rows already satisfied both
   targets. A stricter rule does not have to change a choice on a tiny discrete set.
5. **Explain:** say what further labeling and independent testing you would need before
   using the cutoff as a real review policy.

If no row meets a requested target, report that failure rather than lowering the target
without saying so. This exercise's two targets both have qualifying rows.

## What did we actually build?

- A first splitting rule that leaves some compound claims intact.
- Provisional support labels from word rules, not a pretrained evidence checker.
- Name/year candidates and relation comparisons that give an editor places to inspect.
- A review-cutoff sweep against declared answer-derived labels.

The useful outcome is a clearer review question, not a claim that the system now detects
hallucinations reliably. A real evaluation needs careful claim boundaries, reviewed
evidence labels and independent cases under the intended retrieval conditions.

Next ask a different question:
[when the system says 80% confident, does that number mean anything dependable?](04-calibration-and-confidence.ipynb)

<a id="concept-recap"></a>

## Concept recap

Ask which exact claim the evidence supports, not whether the whole answer sounds convincing.
Keep **supported**, **contradicted** and **unknown** distinct.
Missing evidence is not proof of global falsehood.

```python
pieces = split_claims(answer)
checks = [(piece, claim_outcome(piece, evidence)) for piece in pieces]
```

This sketch uses the earlier toy functions. Splitting at punctuation is not a reliable
claim extractor, and word coverage is not entailment.
Names and years can match while the relationship changes.

A review cutoff trades missed problems against review work:

```python
send_to_review = risk_scores >= threshold
precision = true_positive / max(true_positive + false_positive, 1)
recall = true_positive / max(true_positive + false_negative, 1)
```

These are calculation sketches, not a calibrated detector.
Use independently reviewed claim labels to measure a real operating point.
Keep the claim, evidence span, label source and review reason together.

**Grounding asks what the supplied evidence supports; confidence asks a different question.**
Continue with [calibration](04-calibration-and-confidence.ipynb).